## 1. TF-IDF + linear SVM

In [34]:
import ast
from pathlib import Path
import json
import re

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    hamming_loss,
    precision_score,
    recall_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.svm import LinearSVC

In [4]:
CSV_PATH = 'voc_masked_10000_tags.csv'
RANDOM_STATE = 42
TOP_K = 4
MIN_TAG_PROB = 0.25

In [15]:
def normalize_text(value):
    """결측값, 줄바꿈, 연속 공백 정리"""
    if pd.isna(value):
        return ""

    text = str(value)

    text = text.replace("\u200b", " ")
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

def parse_tags(value):
    """
    CSV의 태그 문자열을 실제 Python 리스트로 변환

    예:
    "['포인트', '미적립']"
        → ['포인트', '미적립']
    """

    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    text = str(value).strip()

    if not text:
        return []

    # ['포인트', '미적립'] 형태
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)

            if isinstance(parsed, list):
                return [
                    normalize_text(tag)
                    for tag in parsed
                    if normalize_text(tag)
                ]

        except (ValueError, SyntaxError):
            pass

    # 혹시 기존 | 또는 쉼표 형태인 경우도 처리
    separator = "|" if "|" in text else ","

    return [
        normalize_text(tag)
        for tag in text.split(separator)
        if normalize_text(tag)
    ]

In [20]:
df = pd.read_csv(CSV_PATH)
df.head()

,voc_id,masked_title,masked_content,large_category,middle_category,small_code,small_category,tags,label_confirmed_yn,masking_review_yn,is_synthetic
0,VOC09805,포인트 적립 기준이 궁금합니다,짧은 거리 주행도 포인트 적립 대상인지 문의드립니다.,포인트,적립기준,S015,포인트 적립 기준 문의,"['문의', '적립기준', '지급조건', '처리상태', '포인트']",Y,Y,Y
1,VOC03814,주행 데이터가 반영되지 않았어요,[날짜]에 [출발지]-[도착지] 구간을 운전했는데 앱에서 기록이 보이지 않습니다. ...,주행,주행기록,S006,주행기록 미반영,"['날짜포함', '문의', '미반영', '위치정보포함', '주행', '주행기록', ...",Y,Y,Y
2,VOC02797,새 비밀번호 등록 문의,비밀번호 찾기 과정에서 [전화번호]로 인증번호가 오지 않습니다. 관련 기준도 함께 ...,회원,계정관리,S005,비밀번호 재설정,"['문의', '본인인증', '비밀번호', '전화번호포함', '회원']",Y,Y,Y
3,VOC06072,포인트가 갑자기 줄었습니다,결제를 취소했는데 차감된 포인트가 복구되지 않았습니다. 처리 방법을 안내해주세요.,포인트,차감,S010,포인트 오차감,"['문의', '오차감', '잔액오류', '포인트']",Y,Y,Y
4,VOC03866,주행기록이 보이지 않습니다,[주행일]에 [출발지]-[도착지] 구간을 운전했는데 앱에서 기록이 보이지 않습니다.,주행,주행기록,S006,주행기록 미반영,"['날짜포함', '미반영', '위치정보포함', '주행', '주행기록']",Y,Y,Y


In [21]:
df['tags'] = df['tags'].apply(parse_tags)
df['text'] = df.apply(lambda row:(
    f'[제목] {row['masked_title']}'
    f'[내용] {row['masked_content']}'), axis=1
)

In [22]:
df.head()

,voc_id,masked_title,masked_content,large_category,middle_category,small_code,small_category,tags,label_confirmed_yn,masking_review_yn,is_synthetic,text
0,VOC09805,포인트 적립 기준이 궁금합니다,짧은 거리 주행도 포인트 적립 대상인지 문의드립니다.,포인트,적립기준,S015,포인트 적립 기준 문의,"[문의, 적립기준, 지급조건, 처리상태, 포인트]",Y,Y,Y,[제목] 포인트 적립 기준이 궁금합니다[내용] 짧은 거리 주행도 포인트 적립 대상인...
1,VOC03814,주행 데이터가 반영되지 않았어요,[날짜]에 [출발지]-[도착지] 구간을 운전했는데 앱에서 기록이 보이지 않습니다. ...,주행,주행기록,S006,주행기록 미반영,"[날짜포함, 문의, 미반영, 위치정보포함, 주행, 주행기록, 처리상태]",Y,Y,Y,[제목] 주행 데이터가 반영되지 않았어요[내용] [날짜]에 [출발지]-[도착지] 구...
2,VOC02797,새 비밀번호 등록 문의,비밀번호 찾기 과정에서 [전화번호]로 인증번호가 오지 않습니다. 관련 기준도 함께 ...,회원,계정관리,S005,비밀번호 재설정,"[문의, 본인인증, 비밀번호, 전화번호포함, 회원]",Y,Y,Y,[제목] 새 비밀번호 등록 문의[내용] 비밀번호 찾기 과정에서 [전화번호]로 인증번...
3,VOC06072,포인트가 갑자기 줄었습니다,결제를 취소했는데 차감된 포인트가 복구되지 않았습니다. 처리 방법을 안내해주세요.,포인트,차감,S010,포인트 오차감,"[문의, 오차감, 잔액오류, 포인트]",Y,Y,Y,[제목] 포인트가 갑자기 줄었습니다[내용] 결제를 취소했는데 차감된 포인트가 복구되...
4,VOC03866,주행기록이 보이지 않습니다,[주행일]에 [출발지]-[도착지] 구간을 운전했는데 앱에서 기록이 보이지 않습니다.,주행,주행기록,S006,주행기록 미반영,"[날짜포함, 미반영, 위치정보포함, 주행, 주행기록]",Y,Y,Y,[제목] 주행기록이 보이지 않습니다[내용] [주행일]에 [출발지]-[도착지] 구간을...


In [6]:
X = df['text']
y = df['small_code']

KeyError: 'text'

In [23]:
# train, valid, test split
train_df, temp_df = train_test_split(df,
                                     test_size=0.2,
                                     random_state=RANDOM_STATE,
                                     stratify=df['small_code'])

valid_df, test_df = train_test_split(temp_df,
                                     test_size=0.5,
                                     random_state=RANDOM_STATE,
                                     stratify=temp_df['small_code'])

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print('train', len(train_df))
print('validation', len(valid_df))
print('test', len(test_df))

train 8000
validation 1000
test 1000


In [26]:
clsf_model = Pipeline(
    steps=[
        (
            'tfidf',
            TfidfVectorizer(
                analyzer='char',
                ngram_range=(2,5),
                min_df=2,
                max_df=0.995,
                max_features=200_200,
                sublinear_tf=True,
                dtype=np.float32,
            ),
        ),
        (
            'classifier',
            LinearSVC(
                C=1.0,
                class_weight='balanced',
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

tag_model = Pipeline(
    steps=[
        (
            'tfidf',
            TfidfVectorizer(
                analyzer='char',
                ngram_range=(2,5),
                min_df=2,
                max_df=0.995,
                max_features=200_200,
                sublinear_tf=True,
                dtype=np.float32,
            ),
        ),
        (
            'tag_classifier',
            OneVsRestClassifier(
                LogisticRegression(
                    C=2.0,
                    max_iter=2_000,
                    class_weight='balanced',
                    solver='liblinear',
                    random_state=RANDOM_STATE,
                ), n_jobs=-1
            ),
        ),
    ]
)

In [27]:
# 태그 정답 multi-hot 변환
mlb = MultiLabelBinarizer()

y_train_tags = mlb.fit_transform(train_df['tags'])
y_valid_tags = mlb.fit_transform(valid_df['tags'])
y_test_tags = mlb.fit_transform(test_df['tags'])
print('MultiLabelbinarizer 태그 순서')
print(mlb.classes_)

MultiLabelbinarizer 태그 순서
['계약정보' '계정삭제' '금액포함' '날짜포함' '로그인' '문의' '미반영' '미적립' '변경요청' '보상문의' '보험'
 '본인인증' '비밀번호' '사건번호포함' '사고' '사용처문의' '삭제요청' '시간포함' '실행오류' '안전운전할인' '앱'
 '앱오류' '오류문의' '오류코드포함' '오차감' '위치정보포함' '잔액오류' '적립기준' '적립지연' '전화번호' '전화번호포함'
 '접속오류' '정보변경' '정보불일치' '주행' '주행기록' '주행포인트' '증권번호포함' '지급조건' '차량' '차량번호포함'
 '차량정보' '처리상태' '처리현황' '포인트' '포인트사용' '할인적용' '확인요청' '회원' '회원탈퇴']


In [30]:
# 모델 학습
print('소분류 예측 모델 학습 시작')
clsf_model.fit(train_df['text'], train_df['small_code'])
print('소분류 예측 모델 학습 완료')

print('태그 추출 모델 학습 시작')
tag_model.fit(train_df['text'], y_train_tags)
print('태그 추출 모델 학습 완료')

소분류 예측 모델 학습 시작
소분류 예측 모델 학습 완료
태그 추출 모델 학습 시작
태그 추출 모델 학습 완료


In [32]:
# 태그 선택 함수
def select_tags(
    probabilities,
    tag_classes,
    top_k=4,
    min_probability=0.25,
):
    """
    확률이 높은 순서대로 최대 top_k개 태그 선택

    min_probability 미만은 제외
    단, 모두 기준 미만이면 가장 높은 태그 1개 반환
    """

    probabilities = np.asarray(probabilities)

    sorted_indices = np.argsort(
        probabilities
    )[::-1]

    selected_tags = []
    selected_details = []

    for index in sorted_indices:
        probability = float(
            probabilities[index]
        )

        if probability < min_probability:
            continue

        tag = str(tag_classes[index])

        selected_tags.append(tag)

        selected_details.append(
            {
                "tag": tag,
                "probability": round(
                    probability,
                    4,
                ),
            }
        )

        if len(selected_tags) >= top_k:
            break

    # 확률 기준을 넘는 태그가 하나도 없는 경우
    if len(selected_tags) == 0:
        best_index = int(
            sorted_indices[0]
        )

        best_tag = str(
            tag_classes[best_index]
        )

        best_probability = float(
            probabilities[best_index]
        )

        selected_tags.append(best_tag)

        selected_details.append(
            {
                "tag": best_tag,
                "probability": round(
                    best_probability,
                    4,
                ),
            }
        )

    return selected_tags, selected_details


def probability_matrix_to_binary(
    probability_matrix,
    top_k=4,
    min_probability=0.25,
):
    """평가를 위해 태그 확률을 0/1 행렬로 변환"""

    binary_matrix = np.zeros_like(
        probability_matrix,
        dtype=int,
    )

    for row_index, probabilities in enumerate(
        probability_matrix
    ):
        sorted_indices = np.argsort(
            probabilities
        )[::-1]

        selected_count = 0

        for column_index in sorted_indices:
            probability = probabilities[
                column_index
            ]

            if probability < min_probability:
                continue

            binary_matrix[
                row_index,
                column_index,
            ] = 1

            selected_count += 1

            if selected_count >= top_k:
                break

        # 하나도 선택되지 않은 경우 가장 높은 태그 1개
        if selected_count == 0:
            best_index = int(
                sorted_indices[0]
            )

            binary_matrix[
                row_index,
                best_index,
            ] = 1

    return binary_matrix


In [35]:
# validation 평가
valid_small_pred = clsf_model.predict(valid_df['text'])

print("\n==============================")
print("Validation 소분류 성능")
print("==============================")

print(
    "Accuracy:",
    round(
        accuracy_score(
            valid_df["small_code"],
            valid_small_pred,
        ),
        4,
    ),
)

print(
    "Macro F1:",
    round(
        f1_score(
            valid_df["small_code"],
            valid_small_pred,
            average="macro",
            zero_division=0,
        ),
        4,
    ),
)

print(
    classification_report(
        valid_df["small_code"],
        valid_small_pred,
        zero_division=0,
    )
)


valid_tag_prob = tag_model.predict_proba(
    valid_df["text"]
)

valid_tag_pred = probability_matrix_to_binary(
    valid_tag_prob,
    top_k=TOP_K,
    min_probability=MIN_TAG_PROB,
)

print("\n==============================")
print("Validation 태그 성능")
print("==============================")

print(
    "Micro Precision:",
    round(
        precision_score(
            y_valid_tags,
            valid_tag_pred,
            average="micro",
            zero_division=0,
        ),
        4,
    ),
)

print(
    "Micro Recall:",
    round(
        recall_score(
            y_valid_tags,
            valid_tag_pred,
            average="micro",
            zero_division=0,
        ),
        4,
    ),
)

print(
    "Micro F1:",
    round(
        f1_score(
            y_valid_tags,
            valid_tag_pred,
            average="micro",
            zero_division=0,
        ),
        4,
    ),
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_valid_tags,
            valid_tag_pred,
            average="macro",
            zero_division=0,
        ),
        4,
    ),
)

print(
    "Subset Accuracy:",
    round(
        accuracy_score(
            y_valid_tags,
            valid_tag_pred,
        ),
        4,
    ),
)

print(
    "Hamming Loss:",
    round(
        hamming_loss(
            y_valid_tags,
            valid_tag_pred,
        ),
        4,
    ),
)


Validation 소분류 성능
Accuracy: 1.0
Macro F1: 1.0
              precision    recall  f1-score   support

        S001       1.00      1.00      1.00        67
        S002       1.00      1.00      1.00        67
        S003       1.00      1.00      1.00        67
        S004       1.00      1.00      1.00        67
        S005       1.00      1.00      1.00        67
        S006       1.00      1.00      1.00        66
        S007       1.00      1.00      1.00        67
        S008       1.00      1.00      1.00        67
        S009       1.00      1.00      1.00        66
        S010       1.00      1.00      1.00        66
        S011       1.00      1.00      1.00        67
        S012       1.00      1.00      1.00        66
        S013       1.00      1.00      1.00        67
        S014       1.00      1.00      1.00        67
        S015       1.00      1.00      1.00        66

    accuracy                           1.00      1000
   macro avg       1.00      1.00

In [36]:
# test 평가
test_small_pred = clsf_model.predict(
    test_df["text"]
)

print("\n==============================")
print("Test 소분류 성능")
print("==============================")

print(
    "Accuracy:",
    round(
        accuracy_score(
            test_df["small_code"],
            test_small_pred,
        ),
        4,
    ),
)

print(
    "Macro F1:",
    round(
        f1_score(
            test_df["small_code"],
            test_small_pred,
            average="macro",
            zero_division=0,
        ),
        4,
    ),
)

print(
    classification_report(
        test_df["small_code"],
        test_small_pred,
        zero_division=0,
    )
)


test_tag_prob = tag_model.predict_proba(
    test_df["text"]
)

test_tag_pred = probability_matrix_to_binary(
    test_tag_prob,
    top_k=TOP_K,
    min_probability=MIN_TAG_PROB,
)

print("\n==============================")
print("Test 태그 성능")
print("==============================")

print(
    "Micro Precision:",
    round(
        precision_score(
            y_test_tags,
            test_tag_pred,
            average="micro",
            zero_division=0,
        ),
        4,
    ),
)

print(
    "Micro Recall:",
    round(
        recall_score(
            y_test_tags,
            test_tag_pred,
            average="micro",
            zero_division=0,
        ),
        4,
    ),
)

print(
    "Micro F1:",
    round(
        f1_score(
            y_test_tags,
            test_tag_pred,
            average="micro",
            zero_division=0,
        ),
        4,
    ),
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_test_tags,
            test_tag_pred,
            average="macro",
            zero_division=0,
        ),
        4,
    ),
)

print(
    "Subset Accuracy:",
    round(
        accuracy_score(
            y_test_tags,
            test_tag_pred,
        ),
        4,
    ),
)

print(
    "Hamming Loss:",
    round(
        hamming_loss(
            y_test_tags,
            test_tag_pred,
        ),
        4,
    ),
)



Test 소분류 성능
Accuracy: 1.0
Macro F1: 1.0
              precision    recall  f1-score   support

        S001       1.00      1.00      1.00        66
        S002       1.00      1.00      1.00        67
        S003       1.00      1.00      1.00        67
        S004       1.00      1.00      1.00        66
        S005       1.00      1.00      1.00        67
        S006       1.00      1.00      1.00        67
        S007       1.00      1.00      1.00        67
        S008       1.00      1.00      1.00        67
        S009       1.00      1.00      1.00        67
        S010       1.00      1.00      1.00        67
        S011       1.00      1.00      1.00        66
        S012       1.00      1.00      1.00        67
        S013       1.00      1.00      1.00        66
        S014       1.00      1.00      1.00        66
        S015       1.00      1.00      1.00        67

    accuracy                           1.00      1000
   macro avg       1.00      1.00      

In [40]:
predicted_tag_tuples = mlb.inverse_transform(
    test_tag_pred
)

result_df = test_df[
    [
        "voc_id",
        "masked_title",
        "masked_content",
        "small_code",
        "small_category",
    ]
].copy()

result_df["true_tags"] = (
    test_df["tags"]
    .apply(str)
)

result_df["predicted_small_code"] = (
    test_small_pred
)

result_df["predicted_tags"] = [
    str(list(tags))
    for tags in predicted_tag_tuples
]

result_df["small_correct_yn"] = np.where(
    result_df["small_code"]
    == result_df["predicted_small_code"],
    "Y",
    "N",
)

In [41]:
result_df

,voc_id,masked_title,masked_content,small_code,small_category,true_tags,predicted_small_code,predicted_tags,small_correct_yn
0,VOC04034,보험료 할인이 반영됐나요,안전운전 점수와 보험료 할인 적용 기준을 안내해주세요.,S007,보험료 할인 적용 문의,"['문의', '보험', '안전운전할인', '처리상태', '할인적용']",S007,"['문의', '보험', '안전운전할인', '할인적용']",Y
1,VOC03521,최근 주행기록 누락 문의,[날짜]에 [출발지]에서 [도착지]까지 구간을 운전했는데 앱에서 기록이 보이지 않습니다. 관련 기준도 함께 알려주세요. 현재 화면에서 직접 수정할 수 없습니다.,S006,주행기록 미반영,"['날짜포함', '문의', '미반영', '변경요청', '위치정보포함', '주행', '주행기록']",S006,"['문의', '미반영', '위치정보포함', '주행']",Y
2,VOC03900,운전한 내역이 앱에 없습니다,주행 종료 후에도 거리와 시간이 0으로 표시됩니다. 필요한 확인 항목이 있다면 안내 부탁드립니다.,S006,주행기록 미반영,"['문의', '미반영', '주행', '주행기록']",S006,"['문의', '미반영', '주행', '주행기록']",Y
3,VOC02139,계정 접속이 불가능합니다 빠른 확인 요청드립니다.,아이디와 비밀번호를 입력해도 로그인 오류가 발생합니다.,S004,로그인 오류,"['로그인', '오류문의', '접속오류', '확인요청', '회원']",S004,"['로그인', '접속오류', '확인요청', '회원']",Y
4,VOC09932,운전점수와 포인트 관계 앱에서 확인이 어려워 문의드립니다.,포인트 지급 대상과 제외 조건을 자세히 안내해주세요.,S015,포인트 적립 기준 문의,"['문의', '적립기준', '지급조건', '포인트']",S015,"['문의', '적립기준', '지급조건', '포인트']",Y
...,...,...,...,...,...,...,...,...,...
995,VOC03978,주행기록이 보이지 않습니다,차량은 정상적으로 운행했는데 주행 데이터가 수집되지 않은 것 같습니다.,S006,주행기록 미반영,"['미반영', '주행', '주행기록']",S006,"['미반영', '주행', '주행기록']",Y
996,VOC06321,포인트 차감 내역이 이상합니다,사용한 적이 없는데 [차감금액] 포인트가 차감된 것으로 표시됩니다.,S010,포인트 오차감,"['금액포함', '오차감', '잔액오류', '포인트']",S010,"['금액포함', '오차감', '잔액오류', '포인트']",Y
997,VOC02606,로그인 화면에서 넘어가지 않아요 빠른 확인 요청드립니다.,여러 번 시도했지만 인증 후 다시 로그인 화면으로 돌아옵니다. 처리 방법을 안내해주세요.,S004,로그인 오류,"['로그인', '문의', '접속오류', '확인요청', '회원']",S004,"['로그인', '접속오류', '확인요청', '회원']",Y
998,VOC09232,휴대전화 번호를 변경하고 싶습니다,새 번호로 본인 인증을 진행하는 방법을 알려주세요.,S014,휴대전화 번호 변경,"['문의', '변경요청', '전화번호', '정보변경', '회원']",S014,"['변경요청', '전화번호', '정보변경', '회원']",Y


In [ ]:
◎■◻︎